In [29]:
import random

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

In [38]:
# Read in and merge data
csv_data = pd.read_csv('../02_processed_data/combined_csv_data.csv')
json_data = pd.read_csv('../02_processed_data/combined_json_data.csv')
all_data = pd.concat([csv_data, json_data])
all_data.drop(['prolificId', 'questionId', 'trialIndex', 'question'], axis=1, inplace=True)

In [92]:
# Process .csv data
pids_to_handle = []
clean_data = pd.DataFrame()
for pid in all_data['participantId'].unique():
    try:
        pid_df = all_data[all_data['participantId'] == pid]
        fixed_choices = pid_df[pid_df['eventType'] == 'fixedChoice']
        disc_trials = pid_df[pid_df['eventType'] == 'discountingTrial']
        for task in disc_trials['taskLabel'].unique():
            task_df = disc_trials[disc_trials['taskLabel'] == task]
            task_df = task_df.drop_duplicates(subset=['probability', 'indifferencePoint']).reset_index(drop=True)
        pid_df = pd.concat([fixed_choices, task_df]).reset_index(drop=True)
        clean_data = pd.concat([clean_data, pid_df])
    except:
        pids_to_handle.append(pid)

clean_data = clean_data.reset_index(drop=True)

In [ ]:
# Process json data
pids_to_handle2 = []
clean_data_2 = pd.DataFrame()
for pid in pids_to_handle:
    pid_df = all_data[all_data['participantId']==pid].reset_index(drop=True)
    for val in list(range(len(pid_df))):
        if pid_df['eventType'][val]=='fixedChoice':
            try: 
                int_true = int(pid_df['optionA'][val][-2:])
                pid_df['taskLabel'][val]="money"
            except:
                pid_df['taskLabel'][val]=pid_df['optionA'][val].split(" ")[-1]
                
        elif pid_df['eventType'][val]=='discountingTrial':
            option = pid_df['optionA'][val].split(" ")[-1]
            try: 
                int_true = int(pid_df['optionA'][val][-2:])
                pid_df['taskLabel'][val]=option + " Discounting"
            except:
                pid_df['taskLabel'][val]=option + "Discounting"

    try:
        fixed_choices = pid_df[pid_df['eventType'] == 'fixedChoice']
        disc_trials = pid_df[pid_df['eventType'] == 'discountingTrial']
        for task in disc_trials['taskLabel'].unique():
            task_df = disc_trials[disc_trials['taskLabel'] == task]
            task_df = task_df.drop_duplicates(subset=['probability', 'indifferencePoint']).reset_index(drop=True)
        pid_df = pd.concat([fixed_choices, task_df]).reset_index(drop=True)
        clean_data_2 = pd.concat([clean_data_2, pid_df])
    except:
        pids_to_handle2.append(pid)

clean_data_2 = clean_data_2.reset_index(drop=True)

/var/folders/c5/vpx80swj4yb9ytlnrk8w2qnr0000gn/T/ipykernel_89553/4117806123.py:10: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  pid_df['taskLabel'][val]="money"
/var/folders/c5/vpx80swj4yb9ytlnrk8w2qnr0000gn/T/ipykernel_89553/4117806123.py:

In [98]:
all_df = pd.concat([clean_data, clean_data_2])
all_df.to_csv('../02_processed_data/combined_data.csv', index=False)